In [1]:
import gc
from itertools import combinations
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import friedmanchisquare, t, wilcoxon
from sklearn.base import clone
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
DATA_DIR = ROOT / "datasets" / "final" / "ml"
MODEL_DIR = ROOT / "models" / "randomized_search"
TRAINING_RESULTS_DIR = ROOT / "results" / "model_training" / "randomized_search"
OUTPUT_DIR = ROOT / "results" / "friedman_test"
FOLD_RESULTS_DIR = OUTPUT_DIR / "folds"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FOLD_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 1
N_SPLITS = 5
N_REPEATS = 30
N_JOBS = -1
ALPHA = 0.05
PRIMARY_METRIC = "f1_macro"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [2]:
TRAINING_CONFIGS = {
    "cicids2017": ["cicids2017"],
    "unsw_nb15": ["unsw_nb15"],
    "iot23": ["iot23"],
    "cicids2017_unsw_nb15": ["cicids2017", "unsw_nb15"],
    "cicids2017_iot23": ["cicids2017", "iot23"],
    "unsw_nb15_iot23": ["unsw_nb15", "iot23"],
}

MODEL_ORDER = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]

TRAINING_SOURCES_TO_RUN = list(TRAINING_CONFIGS)
MODELS_TO_RUN = MODEL_ORDER.copy()

scoring = {
    "precision_attack": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_attack": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1_attack": make_scorer(f1_score, pos_label=1, zero_division=0),
    "precision_benign": make_scorer(precision_score, pos_label=0, zero_division=0),
    "recall_benign": make_scorer(recall_score, pos_label=0, zero_division=0),
    "f1_benign": make_scorer(f1_score, pos_label=0, zero_division=0),
    "f1_macro": make_scorer(
        f1_score,
        average="macro",
        labels=[0, 1],
        zero_division=0,
    ),
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
}


In [3]:
summary_files = [
    TRAINING_RESULTS_DIR / model_name / "summary.csv"
    for model_name in MODEL_ORDER
]

missing_summary_files = [
    str(summary_path)
    for summary_path in summary_files
    if not summary_path.exists()
]
if missing_summary_files:
    raise FileNotFoundError(
        "Resumos de treinamento não encontrados: "
        + ", ".join(missing_summary_files)
    )

training_summary = pd.concat(
    [pd.read_csv(summary_path) for summary_path in summary_files],
    ignore_index=True,
)

expected_experiments = len(MODEL_ORDER) * len(TRAINING_CONFIGS)
assert len(training_summary) == expected_experiments
assert not training_summary.duplicated(["model", "training_source"]).any()
assert set(training_summary["selection_metric"]) == {PRIMARY_METRIC}

selected_experiments = training_summary.loc[
    training_summary["model"].isin(MODELS_TO_RUN)
    & training_summary["training_source"].isin(TRAINING_SOURCES_TO_RUN)
].copy()

model_files = {
    (model_name, training_source): (
        MODEL_DIR / model_name / f"{training_source}.joblib"
    )
    for model_name in MODELS_TO_RUN
    for training_source in TRAINING_SOURCES_TO_RUN
}
missing_model_files = [
    str(model_path)
    for model_path in model_files.values()
    if not model_path.exists()
]
if missing_model_files:
    raise FileNotFoundError(
        "Modelos vencedores não encontrados: "
        + ", ".join(missing_model_files)
    )

display(selected_experiments[[
    "model",
    "training_source",
    "selection_metric",
    "params",
]])


,model,training_source,selection_metric,params
0,decision_tree,cicids2017,f1_macro,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": null, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 1, ""classifier__min_samples_split"": 2, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}"
1,decision_tree,unsw_nb15,f1_macro,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 10, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 2, ""classifier__min_samples_split"": 10, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}"
2,decision_tree,iot23,f1_macro,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 8, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 3, ""classifier__min_samples_split"": 6, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}"
3,decision_tree,cicids2017_unsw_nb15,f1_macro,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 15, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 4, ""classifier__min_samples_split"": 17, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}"
4,decision_tree,cicids2017_iot23,f1_macro,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": null, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 1, ""classifier__min_samples_split"": 2, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}"
5,decision_tree,unsw_nb15_iot23,f1_macro,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 15, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 4, ""classifier__min_samples_split"": 17, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}"
6,random_forest,cicids2017,f1_macro,"{""classifier__bootstrap"": true, ""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": ""balanced"", ""classifier__criterion"": ""gini"", ""classifier__max_depth"": null, ""classifier__max_features"": ""sqrt"", ""classifier__max_leaf_nodes"": null, ""classifier__max_samples"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 2, ""classifier__min_samples_split"": 2, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__n_estimators"": 171, ""classifier__n_jobs"": 1, ""classifier__oob_score"": false, ""classifier__random_state"": 1, ""classifier__verbose"": 0, ""classifier__warm_start"": false}"
7,random_forest,unsw_nb15,f1_macro,"{""classifier__bootstrap"": true, ""classifi

In [4]:
reference_pipeline = joblib.load(model_files[(MODEL_ORDER[0], TRAINING_SOURCES_TO_RUN[0])])
preprocessor = reference_pipeline.named_steps["preprocessor"]
numerical_features = list(preprocessor.transformers[0][2])
categorical_features = list(preprocessor.transformers[1][2])
feature_columns = numerical_features + categorical_features
del reference_pipeline, preprocessor

repeated_cross_validation = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)

def load_training_data(dataset_names):
    columns_to_load = feature_columns + ["label_binary", "dataset_name"]
    dataframes = [
        pd.read_parquet(
            DATA_DIR / f"{dataset_name}_train.parquet",
            columns=columns_to_load,
        )
        for dataset_name in dataset_names
    ]
    training_df = pd.concat(dataframes, ignore_index=True)
    features = training_df[feature_columns]
    target = training_df["label_binary"].eq("ATTACK").astype("int8")
    stratification_labels = (
        training_df["dataset_name"].astype("string")
        + "__"
        + target.astype("string")
    )
    cv_splits = list(
        repeated_cross_validation.split(features, stratification_labels)
    )
    return features, target, cv_splits


def fold_result_path(model_name, training_source):
    model_output_dir = FOLD_RESULTS_DIR / model_name
    model_output_dir.mkdir(parents=True, exist_ok=True)
    return model_output_dir / f"{training_source}.csv"


def result_is_complete(result_path):
    if not result_path.exists():
        return False
    result = pd.read_csv(result_path)
    expected_rows = N_SPLITS * N_REPEATS
    required_columns = {
        "model",
        "training_source",
        "repeat",
        "fold",
        "n_splits",
        "n_repeats",
        "random_state",
        PRIMARY_METRIC,
    }
    if len(result) != expected_rows:
        return False
    if not required_columns.issubset(result.columns):
        return False
    return (
        result["n_splits"].eq(N_SPLITS).all()
        and result["n_repeats"].eq(N_REPEATS).all()
        and result["random_state"].eq(RANDOM_STATE).all()
        and not result[["repeat", "fold"]].duplicated().any()
        and not result[PRIMARY_METRIC].isna().any()
    )


def evaluate_selected_model(model_name, training_source, features, target, cv_splits):
    selected_pipeline = clone(joblib.load(model_files[(model_name, training_source)]))
    evaluation_start = perf_counter()
    cv_results = cross_validate(
        selected_pipeline,
        features,
        target,
        scoring=scoring,
        cv=cv_splits,
        n_jobs=N_JOBS,
        return_train_score=False,
        error_score="raise",
    )
    elapsed_seconds = perf_counter() - evaluation_start
    result = pd.DataFrame({
        "model": model_name,
        "training_source": training_source,
        "repeat": np.repeat(np.arange(1, N_REPEATS + 1), N_SPLITS),
        "fold": np.tile(np.arange(1, N_SPLITS + 1), N_REPEATS),
        "n_splits": N_SPLITS,
        "n_repeats": N_REPEATS,
        "random_state": RANDOM_STATE,
        "fit_time_seconds": cv_results["fit_time"],
        "score_time_seconds": cv_results["score_time"],
    })
    for metric_name in scoring:
        result[metric_name] = cv_results[f"test_{metric_name}"]
    result["evaluation_elapsed_seconds"] = elapsed_seconds
    return result


In [5]:
total_start = perf_counter()
completed_experiments = 0
total_experiments = len(TRAINING_SOURCES_TO_RUN) * len(MODELS_TO_RUN)

for training_source in TRAINING_SOURCES_TO_RUN:
    pending_models = [
        model_name
        for model_name in MODELS_TO_RUN
        if not result_is_complete(fold_result_path(model_name, training_source))
    ]

    if pending_models:
        features, target, cv_splits = load_training_data(
            TRAINING_CONFIGS[training_source]
        )

        for model_name in MODELS_TO_RUN:
            result_path = fold_result_path(model_name, training_source)
            completed_experiments += 1

            if result_is_complete(result_path):
                print(
                    f"{completed_experiments:02d}/{total_experiments} - "
                    f"{model_name} - {training_source} - já concluído"
                )
                continue

            result = evaluate_selected_model(
                model_name,
                training_source,
                features,
                target,
                cv_splits,
            )
            result.to_csv(result_path, index=False)
            elapsed_minutes = result["evaluation_elapsed_seconds"].iloc[0] / 60
            print(
                f"{completed_experiments:02d}/{total_experiments} - "
                f"{model_name} - {training_source} - "
                f"{elapsed_minutes:.2f} min"
            )

        del features, target, cv_splits
        gc.collect()
    else:
        completed_experiments += len(MODELS_TO_RUN)
        print(f"{training_source} - todos os modelos já concluídos")

total_elapsed_seconds = perf_counter() - total_start
print(f"tempo desta execução: {total_elapsed_seconds / 60:.2f} min")


01/30 - decision_tree - cicids2017 - 0.66 min
02/30 - random_forest - cicids2017 - 17.02 min
03/30 - xgboost - cicids2017 - 2.25 min
04/30 - linear_svm - cicids2017 - 4.00 min
05/30 - mlp - cicids2017 - 11.03 min
06/30 - decision_tree - unsw_nb15 - 0.55 min
07/30 - random_forest - unsw_nb15 - 16.87 min
08/30 - xgboost - unsw_nb15 - 2.23 min
09/30 - linear_svm - unsw_nb15 - 2.90 min
10/30 - mlp - unsw_nb15 - 17.49 min
11/30 - decision_tree - iot23 - 0.31 min
12/30 - random_forest - iot23 - 11.32 min
13/30 - xgboost - iot23 - 1.80 min
14/30 - linear_svm - iot23 - 1.54 min
15/30 - mlp - iot23 - 6.06 min
16/30 - decision_tree - cicids2017_unsw_nb15 - 1.44 min
17/30 - random_forest - cicids2017_unsw_nb15 - 45.80 min
18/30 - xgboost - cicids2017_unsw_nb15 - 5.08 min
19/30 - linear_svm - cicids2017_unsw_nb15 - 7.24 min
20/30 - mlp - cicids2017_unsw_nb15 - 55.36 min
21/30 - decision_tree - cicids2017_iot23 - 1.21 min
22/30 - random_forest - cicids2017_iot23 - 25.98 min
23/30 - xgboost - cicids

In [6]:
complete_training_sources = []
fold_frames = []

for training_source in TRAINING_SOURCES_TO_RUN:
    source_paths = [
        fold_result_path(model_name, training_source)
        for model_name in MODELS_TO_RUN
    ]
    if all(result_is_complete(result_path) for result_path in source_paths):
        complete_training_sources.append(training_source)
        fold_frames.extend(pd.read_csv(result_path) for result_path in source_paths)

if not fold_frames:
    raise RuntimeError("nenhuma origem de treinamento está completa")

fold_results = pd.concat(fold_frames, ignore_index=True)
fold_results = fold_results.sort_values(
    ["training_source", "model", "repeat", "fold"]
).reset_index(drop=True)
fold_results.to_csv(OUTPUT_DIR / "fold_results.csv", index=False)

aggregation = {metric_name: "mean" for metric_name in scoring}
aggregation.update({
    "fit_time_seconds": "sum",
    "score_time_seconds": "sum",
})
repeat_results = (
    fold_results.groupby(
        ["training_source", "model", "repeat"],
        as_index=False,
    )
    .agg(aggregation)
)
repeat_results.to_csv(OUTPUT_DIR / "repeat_results.csv", index=False)

def confidence_interval_95(values):
    values = pd.Series(values).dropna()
    standard_error = values.std(ddof=1) / np.sqrt(len(values))
    margin = t.ppf(0.975, df=len(values) - 1) * standard_error
    return pd.Series({
        "f1_macro_mean": values.mean(),
        "f1_macro_std": values.std(ddof=1),
        "f1_macro_min": values.min(),
        "f1_macro_max": values.max(),
        "f1_macro_ci95_lower": values.mean() - margin,
        "f1_macro_ci95_upper": values.mean() + margin,
        "n_repeats": len(values),
    })


stability_summary = (
    repeat_results.groupby(["training_source", "model"])[PRIMARY_METRIC]
    .apply(confidence_interval_95)
    .unstack()
    .reset_index()
)
stability_summary.to_csv(OUTPUT_DIR / "stability_summary.csv", index=False)

display(stability_summary)


,training_source,model,f1_macro_mean,f1_macro_std,f1_macro_min,f1_macro_max,f1_macro_ci95_lower,f1_macro_ci95_upper,n_repeats
0,cicids2017,decision_tree,0.990408,0.000157,0.990111,0.990714,0.990350,0.990467,30.0
1,cicids2017,linear_svm,0.903022,0.000621,0.902190,0.904159,0.902790,0.903254,30.0
2,cicids2017,mlp,0.990556,0.000531,0.987901,0.990897,0.990358,0.990754,30.0
3,cicids2017,random_forest,0.992492,0.000086,0.992306,0.992616,0.992460,0.992524,30.0
4,cicids2017,xgboost,0.991770,0.000078,0.991645,0.991982,0.991741,0.991799,30.0
5,cicids2017_iot23,decision_tree,0.993675,0.000067,0.993568,0.993798,0.993650,0.993700,30.0
6,cicids2017_iot23,linear_svm,0.948489,0.000105,0.948286,0.948817,0.948449,0.948528,30.0
7,cicids2017_iot23,mlp,0.992600,0.000289,0.991200,0.992886,0.992492,0.992708,30.0
8,cicids2017_iot23,random_forest,0.994579,0.000056,0.994437,0.994740,0.994558,0.994600,30.0
9,cicids2017_iot23,xgboost,0.994182,0.000055,0.994056,0.994289,0.994161,0.994203,30.0


In [7]:
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values), dtype=float)
    running_maximum = 0.0

    for position, original_index in enumerate(order):
        multiplier = len(p_values) - position
        corrected_value = multiplier * p_values[original_index]
        running_maximum = max(running_maximum, corrected_value)
        adjusted[original_index] = min(running_maximum, 1.0)

    return adjusted


friedman_rows = []
friedman_pivots = {}

for training_source in complete_training_sources:
    source_results = repeat_results.loc[
        repeat_results["training_source"].eq(training_source)
    ]
    pivot = source_results.pivot(
        index="repeat",
        columns="model",
        values=PRIMARY_METRIC,
    ).reindex(columns=MODEL_ORDER)

    assert pivot.shape == (N_REPEATS, len(MODEL_ORDER))
    assert not pivot.isna().any().any()

    statistic, p_value = friedmanchisquare(
        *[pivot[model_name].to_numpy() for model_name in MODEL_ORDER]
    )
    ranks = pivot.rank(axis=1, ascending=False, method="average")
    row = {
        "training_source": training_source,
        "metric": PRIMARY_METRIC,
        "n_models": len(MODEL_ORDER),
        "n_blocks": len(pivot),
        "statistic": statistic,
        "p_value": p_value,
        "kendalls_w": statistic / (len(pivot) * (len(MODEL_ORDER) - 1)),
    }
    for model_name in MODEL_ORDER:
        row[f"mean_{model_name}"] = pivot[model_name].mean()
        row[f"mean_rank_{model_name}"] = ranks[model_name].mean()
    friedman_rows.append(row)
    friedman_pivots[training_source] = pivot

friedman_results = pd.DataFrame(friedman_rows)
friedman_results["p_value_holm"] = holm_adjust(friedman_results["p_value"])
friedman_results["significant"] = friedman_results["p_value_holm"].lt(ALPHA)
friedman_results.to_csv(OUTPUT_DIR / "friedman_results.csv", index=False)

display(friedman_results)


,training_source,metric,n_models,n_blocks,statistic,p_value,kendalls_w,mean_decision_tree,mean_rank_decision_tree,mean_random_forest,mean_rank_random_forest,mean_xgboost,mean_rank_xgboost,mean_linear_svm,mean_rank_linear_svm,mean_mlp,mean_rank_mlp,p_value_holm,significant
0,cicids2017,f1_macro,5,30,117.840000,1.545044e-24,0.982000,0.990408,3.900000,0.992492,1.000000,0.991770,2.000000,0.903022,5.0,0.990556,3.100000,4.635133e-24,True
1,unsw_nb15,f1_macro,5,30,118.506667,1.113231e-24,0.987556,0.979775,3.000000,0.980831,1.933333,0.980997,1.066667,0.926475,5.0,0.979259,4.000000,4.452925e-24,True
2,iot23,f1_macro,5,30,114.186667,9.306935e-24,0.951556,0.997698,3.033333,0.997808,1.700000,0.997829,1.300000,0.995747,5.0,0.997321,3.966667,1.073490e-23,True
3,cicids2017_unsw_nb15,f1_macro,5,30,120.000000,5.341472e-25,1.000000,0.983371,3.000000,0.986147,1.000000,0.985626,2.000000,0.844980,5.0,0.980448,4.000000,3.204883e-24,True
4,cicids2017_iot23,f1_macro,5,30,120.000000,5.341472e-25,1.000000,0.993675,3.000000,0.994579,1.000000,0.994182,2.000000,0.948489,5.0,0.992600,4.000000,3.204883e-24,True
5,unsw_nb15_iot23,f1_macro,5,30,115.306667,5.367452e-24,0.960889,0.988255,3.000000,0.989195,1.733333,0.989216,1.266667,0.955032,5.0,0.986596,4.000000,1.073490e-23,True


In [8]:
posthoc_rows = []

for friedman_result in friedman_results.itertuples(index=False):
    if not friedman_result.significant:
        continue

    pivot = friedman_pivots[friedman_result.training_source]
    source_rows = []

    for model_a, model_b in combinations(MODEL_ORDER, 2):
        values_a = pivot[model_a].to_numpy()
        values_b = pivot[model_b].to_numpy()

        if np.allclose(values_a, values_b):
            statistic = 0.0
            p_value = 1.0
        else:
            statistic, p_value = wilcoxon(
                values_a,
                values_b,
                alternative="two-sided",
                method="auto",
            )

        source_rows.append({
            "training_source": friedman_result.training_source,
            "metric": PRIMARY_METRIC,
            "model_a": model_a,
            "model_b": model_b,
            "mean_a": values_a.mean(),
            "mean_b": values_b.mean(),
            "mean_difference_a_minus_b": values_a.mean() - values_b.mean(),
            "statistic": statistic,
            "p_value": p_value,
        })

    adjusted_p_values = holm_adjust([row["p_value"] for row in source_rows])
    for row, adjusted_p_value in zip(source_rows, adjusted_p_values):
        row["p_value_holm"] = adjusted_p_value
        row["significant"] = adjusted_p_value < ALPHA
        posthoc_rows.append(row)

posthoc_columns = [
    "training_source",
    "metric",
    "model_a",
    "model_b",
    "mean_a",
    "mean_b",
    "mean_difference_a_minus_b",
    "statistic",
    "p_value",
    "p_value_holm",
    "significant",
]
posthoc_results = pd.DataFrame(posthoc_rows, columns=posthoc_columns)
posthoc_results.to_csv(OUTPUT_DIR / "posthoc_wilcoxon_holm.csv", index=False)

if posthoc_results.empty:
    print("nenhum pós-hoc foi necessário")
else:
    display(posthoc_results)


,training_source,metric,model_a,model_b,mean_a,mean_b,mean_difference_a_minus_b,statistic,p_value,p_value_holm,significant
0,cicids2017,f1_macro,decision_tree,random_forest,0.990408,0.992492,-0.002084,0.0,1.862645e-09,1.862645e-08,True
1,cicids2017,f1_macro,decision_tree,xgboost,0.990408,0.991770,-0.001362,0.0,1.862645e-09,1.862645e-08,True
2,cicids2017,f1_macro,decision_tree,linear_svm,0.990408,0.903022,0.087386,0.0,1.862645e-09,1.862645e-08,True
3,cicids2017,f1_macro,decision_tree,mlp,0.990408,0.990556,-0.000147,51.0,6.286614e-05,6.286614e-05,True
4,cicids2017,f1_macro,random_forest,xgboost,0.992492,0.991770,0.000722,0.0,1.862645e-09,1.862645e-08,True
5,cicids2017,f1_macro,random_forest,linear_svm,0.992492,0.903022,0.089470,0.0,1.862645e-09,1.862645e-08,True
6,cicids2017,f1_macro,random_forest,mlp,0.992492,0.990556,0.001936,0.0,1.862645e-09,1.862645e-08,True
7,cicids2017,f1_macro,xgboost,linear_svm,0.991770,0.903022,0.088748,0.0,1.862645e-09,1.862645e-08,True
8,cicids2017,f1_macro,xgboost,mlp,0.991770,0.990556,0.001214,0.0,1.862645e-09,1.862645e-08,True
9,cicids2017,f1_macro,linear_svm,mlp,0.903022,0.990556,-0.087534,0.0,1.862645e-09,1.862645e-08,True
